# Day 3 Assignment: CTAS and Table Metadata
## Scenario: Cyntexa Logistics Data Ingestion


## Task 1: Ingest CSV using CTAS with read_files()


In [0]:
%sql
-- Step 1: Create a schema (database) to keep things organized
CREATE SCHEMA IF NOT EXISTS cyntexa_dev.cyntexa_logistics;

-- Step 2: Use CTAS to read the CSV file and create a Delta table
-- read_files() reads the CSV directly in the SQL query
CREATE OR REPLACE TABLE cyntexa_dev.cyntexa_logistics.shipments_raw
AS SELECT * FROM read_files(
    '/Volumes/cyntexa_dev/cyntexa_logistics/raw/Shipments/',
    format => 'csv',
    header => 'true',
    inferSchema => 'true'
);

-- Step 3: Check the data
SELECT * FROM cyntexa_dev.cyntexa_logistics.shipments_raw LIMIT 10;

## Task 2: Ingest Nested JSON with Flattened Columns

In [0]:
%sql
-- Step 1: Create the table using CTAS + read_files()
CREATE OR REPLACE TABLE cyntexa_dev.cyntexa_logistics.shipments_json
AS SELECT 
-- Top-level fields
shipment_id,
-- Nested field 1: sender.name (dot notation pulls it out)
sender.name AS sender_name,
-- Nested field 2: sender.address.city (two levels deep!)
sender.address.city AS sender_city,
-- Nested field 3: sender.address.zip
sender.address.zip AS sender_zip,
-- Keep the original nested structure too (for provenance/tracing)
sender AS sender_full_json,
items AS item_array
FROM read_files(
    '/Volumes/cyntexa_dev/cyntexa_logistics/raw/Shipments Json/',
    format => 'json',
    multiLine => 'true'
);


-- Step 2: Verify the flattened columns
SELECT shipment_id , sender_name , sender_city , sender_zip FROM cyntexa_dev.cyntexa_logistics.shipments_json LIMIT 10;

## Task 3: Table Metadata with DESCRIBE Commands

### 3.1 DESCRIBE TABLE


In [0]:
%sql
-- --------------------------------------------------
-- 3.1 DESCRIBE TABLE (The Basic Label)
-- --------------------------------------------------

-- Shows: column names, data types, and comments
DESCRIBE TABLE cyntexa_dev.cyntexa_logistics.shipments_raw;


### 3.2 DESCRIBE EXTENDED

In [0]:
%sql
-- --------------------------------------------------
-- 3.2 DESCRIBE EXTENDED (The Full Ingredient List)
-- --------------------------------------------------

-- Shows: everything from DESCRIBE TABLE PLUS:
--   - Table Type (MANAGED or EXTERNAL)
--   - Location (where files are stored)
--   - Provider (Delta, Parquet, etc.)
--   - Properties and comments
DESCRIBE EXTENDED cyntexa_dev.cyntexa_logistics.shipments_raw;

### 3.3 DESCRIBE DETAIL

In [0]:
%sql
-- --------------------------------------------------
-- 3.3 DESCRIBE DETAIL (The Nutrition Facts)
-- --------------------------------------------------
-- Shows: file-level statistics
--   - Number of files
--   - Total size (bytes)
--   - Number of rows
--   - Partition columns
--   - Min/Max data versions (Delta history)
DESCRIBE DETAIL cyntexa_dev.cyntexa_logistics.shipments_raw;

## Observations & Notes

%md
## Observations & Notes

### Task 1: CSV Ingestion
- `read_files()` made it easy to ingest CSV directly in SQL without writing Python.
- `inferSchema` correctly guessed STRING, INT, DATE types.
- Table is MANAGED (Databricks controls the files).

### Task 2: JSON Flattening
- Used dot notation (`sender.address.city`) to pull nested fields to top level.
- Kept original `sender` column for provenance — if a row is bad, we can trace it back.
- `multiLine =&gt; 'true'` was needed because JSON objects spanned multiple lines.

### Task 3: Metadata Comparison
| Command | Unique Info |
|---------|-------------|
| DESCRIBE | Column names and data types only |
| DESCRIBE EXTENDED | Table type, storage location, provider, properties |
| DESCRIBE DETAIL | File count, size, row count, partitions, history |

### Key Takeaway
CTAS + read_files() is a fast, one-step way to ingest files into Delta tables with full schema inference. Keeping raw nested data alongside flattened columns ensures provenance.

# Intermediate Tasks: Combining Concepts & Making Judgment Calls

## Task 4: Add Provenance with _metadata Columns

In [0]:
%sql
-- ============================================================
-- TASK 4: Add _metadata.file_name and _metadata.file_path
-- ============================================================

-- Step 1: Create a new table WITH provenance columns
-- _metadata is a hidden "shipping label" on every row
CREATE OR REPLACE TABLE cyntexa_dev.cyntexa_logistics.shipments_with_source
AS SELECT 
-- All the original columns from the CSV
*, 
-- PROVENANCE COLUMN 1: Just the file name (like "shipments.csv")
_metadata.file_name AS source_file_name,
-- PROVENANCE COLUMN 2: Full path (like "/FileStore/tables/shipments.csv")
_metadata.file_path AS source_file_path,
-- BONUS: Also grab when the file was last modified
_metadata.file_modification_time AS source_file_modified_time
FROM read_files(
    '/Volumes/cyntexa_dev/cyntexa_logistics/raw/Shipments/shipments.csv',
    format => 'csv',
    header => 'true',
    inferSchema => 'true'
);

-- Step 2: Prove it works — show that every row knows its parent file
SELECT shipment_id , origin_city , destination_city , source_file_name , source_file_path , source_file_modified_time
FROM cyntexa_dev.cyntexa_logistics.shipments_with_source LIMIT 5;

## Task 5: Create an Iceberg Table & Compare with Delta

In [0]:
%sql
-- ============================================================
-- TASK 5: Create an Iceberg Table from the Same Source
-- ============================================================

-- Step 1: Create the SAME table but in ICEBERG format
-- USING ICEBERG tells Databricks: "Use the Iceberg format, not Delta"
CREATE OR REPLACE TABLE cyntexa_dev.cyntexa_logistics.shipments_iceberg USING ICEBERG 
AS SELECT * FROM read_files(
    '/Volumes/cyntexa_dev/cyntexa_logistics/raw/Shipments/shipments.csv',
    format => 'csv',
    header => 'true',
    inferSchema => 'true'
);

### Code: Compare DESCRIBE DETAIL — Delta vs Iceberg

In [0]:
%sql
-- Step 2: Describe the DELTA table (from Task 1)
DESCRIBE DETAIL cyntexa_dev.cyntexa_logistics.shipments_raw;

In [0]:
%sql
-- Step 3: Describe the ICEBERG table
DESCRIBE DETAIL cyntexa_dev.cyntexa_logistics.shipments_iceberg;

In [0]:
%sql
-- ============================================================
-- TASK 6: Records Per Source File — Vendor Audit Report
-- ============================================================

-- Step 1: Build the audit report using the provenance columns
-- from Task 4. Group by file name and COUNT rows.
SELECT 
-- The file name
source_file_name,
-- The full path
source_file_path,
-- How many rows came from THIS file?
COUNT(*) AS record_count,
-- When was this file last touched?
MAX(source_file_modified_time) AS file_last_modified,
-- Bonus: earliest and latest dates in the file (data quality check)
MIN(ship_date) AS earliest_shipment,
MAX(ship_date) AS latest_shipment
FROM cyntexa_dev.cyntexa_logistics.shipments_with_source
-- Group BY file means: "Give me one row per source file"
GROUP BY source_file_name , source_file_path
ORDER BY source_file_name;

## Intermediate Summary & Key Takeaways

### Task 4: Provenance
- `_metadata` columns act like a **shipping label** on every row
- `_metadata.file_name` and `_metadata.file_path` let you trace bad data back to its source
- This is critical for **data lineage** and debugging

### Task 5: Delta vs Iceberg
- **Delta** = Databricks-native, fast, feature-rich
- **Iceberg** = Open standard, works across many tools (Spark, Trino, Flink)
- Both store the same data but use different **metadata formats**
- `DESCRIBE DETAIL` shows `format='delta'` vs `format='iceberg'`

### Task 6: Audit Reports
- Group by `source_file_name` and `COUNT(*)` to verify vendor file drops
- Compare `record_count` against what the vendor claims
- Add `MIN()` / `MAX()` dates as **sanity checks** for data quality

# Advanced Tasks: Production-Ready Decisions

## Task 7: Handle Multiple CSV Files with Different Formats

In [0]:
%sql
-- ============================================================
-- FILE 1: Standard format (comma, all expected columns)
-- ============================================================
CREATE OR REPLACE TABLE cyntexa_dev.cyntexa_logistics.file_standard
AS SELECT *,
_metadata.file_name AS source_file,
_metadata.file_path AS source_path
FROM read_files(
    '/Volumes/cyntexa_dev/cyntexa_logistics/raw/Shipments Different Formats/Standard/',
    format => 'csv',
    header => 'true',
    inferSchema => 'true',
    delimiter => ','       -- comma is the default, but being explicit helps
);

-- checking our data 
SELECT * FROM cyntexa_dev.cyntexa_logistics.file_standard;

In [0]:
%sql
-- ============================================================
-- FILE 2: Pipe-delimited + extra 'priority' column
-- ============================================================
CREATE OR REPLACE TABLE cyntexa_dev.cyntexa_logistics.file_pipe
AS SELECT *,
_metadata.file_name AS source_file,
_metadata.file_path AS source_path
FROM read_files(
    '/Volumes/cyntexa_dev/cyntexa_logistics/raw/Shipments Different Formats/Pipe Delimited /',
    format => 'csv',
    header => 'true',
    inferSchema => 'true',
    delimiter => '|'         -- <-- THIS CHANGED! Pipe instead of comma
);  

-- checking our data
SELECT * FROM cyntexa_dev.cyntexa_logistics.file_pipe;

In [0]:
%sql
-- ============================================================
-- FILE 3: Tab-delimited + missing 'destination' + extra 'carrier'
-- ============================================================
CREATE OR REPLACE TABLE cyntexa_dev.cyntexa_logistics.file_tab
AS SELECT *,
_metadata.file_name AS source_file,
_metadata.file_path AS source_path
FROM read_files(
    '/Volumes/cyntexa_dev/cyntexa_logistics/raw/Shipments Different Formats/Tab Delimited/shipments_tab_delimited.csv',
    format => 'csv',
    header => 'true',
    inferSchema => 'true',
    delimiter => '\t'  -- <-- THIS CHANGED! Tab character
);

-- checking our data
SELECT * FROM cyntexa_dev.cyntexa_logistics.file_tab;

In [0]:
%sql
-- ============================================================
-- FILE 4: Semicolon-delimited + DD-MM-YYYY date format + extra 'cost'
-- ============================================================

CREATE OR REPLACE TABLE cyntexa_dev.cyntexa_logistics.file_semicolon
AS SELECT *,
_metadata.file_name AS source_file,
_metadata.file_path AS source_path
FROM read_files(
    '/Volumes/cyntexa_dev/cyntexa_logistics/raw/Shipments Different Formats/Semicolon/',
    format => 'csv',
    header => 'true',
    inferSchema => 'true',
    delimiter => ';'   -- <-- THIS CHANGED! Semicolon
);

-- checking our data
SELECT * FROM cyntexa_dev.cyntexa_logistics.file_semicolon;

In [0]:
%sql
-- first go the correct catalog
USE CATALOG cyntexa_dev;

-- select the correct schema   
USE SCHEMA cyntexa_logistics;

In [0]:
%sql
SELECT current_catalog() AS my_catalog, current_schema() AS my_schema;


In [0]:
%sql
-- ============================================================
-- DETECTION STRATEGY: Schema Validation Query
-- Run this BEFORE merging files into a unified table
-- ============================================================

-- Step 1: Create a "rule book" — what columns SHOULD exist?
CREATE OR REPLACE TABLE cyntexa_dev.cyntexa_logistics.expected_schema(
    column_name STRING,
    is_required BOOLEAN,
    expected_type STRING
);

INSERT INTO cyntexa_dev.cyntexa_logistics.expected_schema VALUES
    ('shipment_id', true, 'STRING'),
    ('origin', true, 'STRING'),
    ('destination', true, 'STRING'),
    ('weight_kg', true, 'DOUBLE'),
    ('ship_date', true, 'DATE');

-- Step 2: Compare actual columns against expected columns
-- This query finds files that are MISSING required columns
SELECT 
t.table_name AS source_table,
e.column_name AS expected_column,
CASE 
WHEN c.column_name IS NULL THEN 'MISSING'
ELSE 'PRESENT'
END AS status 
FROM cyntexa_dev.cyntexa_logistics.expected_schema e 
CROSS JOIN (
    SELECT 'file_standard' AS table_name UNION ALL
    SELECT 'file_pipe' UNION ALL
    SELECT 'file_tab' UNION ALL
    SELECT 'file_semicolon'
) t
LEFT JOIN information_schema.columns c
    ON c.column_name = e.column_name
    AND c.table_name = t.table_name
    AND c.table_catalog = 'cyntexa_dev'
    AND c.table_schema = 'cyntexa_logistics'  
ORDER BY source_table, expected_column;

In [0]:
%sql
-- ============================================================
-- UNIFIED INGESTION: Combine all files into one clean table
-- ============================================================

CREATE OR REPLACE TABLE cyntexa_logistics.all_shipments_unified
AS
-- Standard file (clean, no fixes needed)
SELECT
    shipment_id,
    origin,
    destination,
    weight_kg,
    TO_DATE(ship_date, 'yyyy-MM-dd') AS ship_date,  -- standard format
    CAST(NULL AS STRING) AS priority,               -- extra column: fill with NULL
    CAST(NULL AS STRING) AS carrier,                -- extra column: fill with NULL
    CAST(NULL AS DOUBLE) AS cost,                   -- extra column: fill with NULL
    source_file,
    source_path
FROM cyntexa_dev.cyntexa_logistics.file_standard

UNION ALL

-- Pipe-delimited file (has 'priority' extra column)
SELECT
    shipment_id,
    origin,
    destination,
    weight_kg,
    TO_DATE(ship_date, 'yyyy-MM-dd') AS ship_date,
    priority,                                         -- exists in this file
    CAST(NULL AS STRING) AS carrier,                -- doesn't exist: NULL
    CAST(NULL AS DOUBLE) AS cost,                   -- doesn't exist: NULL
    source_file,
    source_path
FROM cyntexa_dev.cyntexa_logistics.file_pipe

UNION ALL

-- Tab-delimited file (missing 'destination', has 'carrier')
SELECT
    shipment_id,
    origin,
    CAST(NULL AS STRING) AS destination,            -- missing: fill with NULL
    weight_kg,
    TO_DATE(ship_date, 'yyyy-MM-dd') AS ship_date,
    CAST(NULL AS STRING) AS priority,               -- doesn't exist: NULL
    carrier,                                        -- exists in this file
    CAST(NULL AS DOUBLE) AS cost,                   -- doesn't exist: NULL
    source_file,
    source_path
FROM cyntexa_dev.cyntexa_logistics.file_tab

UNION ALL

-- Semicolon-delimited file (different date format, has 'cost')
SELECT
    shipment_id,
    origin,
    destination,
    weight_kg,
    TO_DATE(ship_date, 'dd-MM-yyyy') AS ship_date,  -- <-- DIFFERENT FORMAT!
    CAST(NULL AS STRING) AS priority,               -- doesn't exist: NULL
    CAST(NULL AS STRING) AS carrier,                -- doesn't exist: NULL
    cost,                                           -- exists in this file
    source_file,
    source_path
FROM cyntexa_dev.cyntexa_logistics.file_semicolon;

-- Verify the unified table
SELECT * FROM cyntexa_dev.cyntexa_logistics.all_shipments_unified;

# 📘 Notes: Delta vs UniForm vs Iceberg (Task 8) 
> **When to choose what for your tables at Cyntexa**

---

## 1. The Three Formats

| Format | Simple Meaning |
|--------|----------------|
| **Native Delta** | Table built for Databricks/Spark only. Fastest here, but other tools struggle to read it. |
| **Delta UniForm** | One table, two "indexes". Still a Delta table, but also creates Iceberg metadata automatically. Other tools can read it like an Iceberg table. |
| **Native Iceberg** | A true Iceberg table. Any engine — Spark, Trino, Snowflake, Flink — can read and write it natively. |

---

## 2. Quick Decision Chart

```
Is Databricks the ONLY tool using this table?
    └── YES → Use NATIVE DELTA
    └── NO  → Does Snowflake/Trino need to WRITE to this table?
                └── YES → Use NATIVE ICEBERG
                └── NO  → Use DELTA UNIFORM
```

---

## 3. When to Use What

### ✅ Native Delta
- Only Databricks writes and reads the table.
- You need **Change Data Feed (CDF)** — track which rows changed when.
- You need **maximum speed** — Photon engine works best here.
- You use **deletion vectors** for fast soft deletes.

### ✅ Delta UniForm
- Databricks writes the table.
- Snowflake or Trino only **reads** the table.
- You want **zero data copies** — same files, dual metadata.
- Downstream tools don't need to write back.

> ⚠️ **Limitations of UniForm:**
> - Iceberg readers can only **read**, not write.
> - Does **not** support tables with deletion vectors enabled.
> - Metadata sync is **asynchronous** — slight delay for Iceberg readers.

### ✅ Native Iceberg
- **Multiple engines** need to write: Spark, Flink, Trino, Snowflake.
- Snowflake needs to **update** the table, not just read it.
- You need **partition evolution** — change how data is partitioned without rewriting files.
- You want to avoid **vendor lock-in** (Iceberg is Apache open-source).

> ⚠️ **Trade-off:** Iceberg in Databricks is ~10-20% slower than Native Delta.

---

## 4. Downstream Tool Support

| Tool | Native Delta | Delta UniForm | Native Iceberg |
|------|:----------:|:-----------:|:------------:|
| **Databricks/Spark** | ⭐⭐⭐ Best | ⭐⭐⭐ Best | ⭐⭐ Good |
| **Snowflake (Read)** | ❌ Hard | ✅ Easy | ✅ Easy |
| **Snowflake (Write)** | ❌ No | ❌ No | ✅ Yes |
| **Trino/Starburst** | ⚠️ Weak | ✅ Good | ⭐⭐⭐ Best |
| **Flink** | ⚠️ Weak | ⚠️ Weak | ✅ Yes |

---

## 5. One-Line Rules to Remember

| Situation | Pick This |
|-----------|-----------|
| Databricks only | **Native Delta** |
| Databricks writes, others read | **Delta UniForm** |
| Everyone reads + writes | **Native Iceberg** |
| Snowflake needs to update data | **Native Iceberg** |
| Trino is your main query engine | **Native Iceberg** |
| Need to change partitions later | **Native Iceberg** |

---

> 💡 **Bottom Line:** Start with Native Delta. If others need to read, add UniForm. If others need to write, switch to Iceberg.


In [0]:
%sql
-- ============================================================
-- ENABLE DELTA UNIFORM (makes Delta table also readable as Iceberg)
-- ============================================================

-- Step 1: Disable deletion vectors (required for IcebergCompatV2)
ALTER TABLE cyntexa_dev.cyntexa_logistics.all_shipments_unified
SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'false');

-- Step 2: Purge deletion vectors from the table
REORG TABLE cyntexa_dev.cyntexa_logistics.all_shipments_unified APPLY (PURGE);

-- Step 3: Enable column mapping mode (required for IcebergCompatV2)
ALTER TABLE cyntexa_dev.cyntexa_logistics.all_shipments_unified
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

-- Step 4: Enable IcebergCompat (required for existing tables)
ALTER TABLE cyntexa_dev.cyntexa_logistics.all_shipments_unified
SET TBLPROPERTIES ('delta.enableIcebergCompatV2' = 'true');

-- Step 5: Enable Universal Format with Iceberg
ALTER TABLE cyntexa_dev.cyntexa_logistics.all_shipments_unified
SET TBLPROPERTIES ('delta.universalFormat.enabledFormats' = 'iceberg');

-- Verify: Check that UniForm is enabled
DESCRIBE EXTENDED cyntexa_dev.cyntexa_logistics.all_shipments_unified;
-- Look for: delta.universalFormat.enabledFormats = iceberg

## Task 9: Trace a Bad Row Using DESCRIBE HISTORY + Metadata

### Step 0: Plant a Bad Row (Create the Problem)


In [0]:
%sql
-- ============================================================
-- STEP 0: PLANT A BAD ROW (Create the problem to solve)
-- ============================================================

-- Insert a row with an impossible weight (9999 kg for a laptop!)
-- Also give it a source_file so we can trace it later
INSERT INTO cyntexa_dev.cyntexa_logistics.all_shipments_unified VALUES
(
    'SH99999',           -- shipment_id
    'New York',          -- origin
    'Los Angeles',       -- destination
    9999.99,             -- weight_kg  <-- THIS IS THE BAD DATA!
    DATE('2026-08-25'),  -- ship_date
    NULL,                -- priority
    NULL,                -- carrier
    NULL,                -- cost
    'shipments_bad_vendor.csv',   -- source_file (the bad file)
    '/Volumes/cyntexa_dev/cyntexa_logistics/raw/Shipments/shipments_bad_vendor.csv'  -- source_path
);

-- Confirm the bad row is now in the table
SELECT * FROM cyntexa_dev.cyntexa_logistics.all_shipments_unified WHERE shipment_id = 'SH99999';

### Step 1: Find the Bad Row

In [0]:
%sql
-- ============================================================
-- STEP 1: FIND THE SUSPICIOUS ROW
-- ============================================================

-- Look for anything that doesn't make sense
-- A laptop or small package shouldn't weigh more than 100 kg!

SELECT *
FROM cyntexa_dev.cyntexa_logistics.all_shipments_unified
WHERE weight_kg > 100;       -- impossible weight for most packages
-- Result: SH99999 shows weight_kg = 9999.99  🚨 ALERT!

### Step 2: Use DESCRIBE HISTORY to Find When It Was Added

In [0]:
%sql
-- ============================================================
-- STEP 2: DESCRIBE HISTORY (Check the security camera)
-- ============================================================

DESCRIBE HISTORY cyntexa_dev.cyntexa_logistics.all_shipments_unified;

### Step 3: Time Travel — Before vs After

In [0]:
%sql
-- ============================================================
-- STEP 3: TIME TRAVEL — See what the table looked like BEFORE
-- ============================================================

-- Look at version just before the bad row 
SELECT * FROM cyntexa_dev.cyntexa_logistics.all_shipments_unified VERSION AS OF 10 WHERE shipment_id = 'SH99999';

In [0]:
%sql
-- Look at the current version 
SELECT * FROM cyntexa_dev.cyntexa_logistics.all_shipments_unified VERSION AS OF 11 WHERE shipment_id = 'SH99999';

In [0]:
%sql
-- Compare side-by-side
SELECT
'(v10)' AS when_checked , 
(
   SELECT COUNT(*) FROM cyntexa_dev.cyntexa_logistics.all_shipments_unified VERSION AS OF 10 WHERE shipment_id = 'SH99999' 
) AS bad_row_count 
UNION ALL 
SELECT 
'(v11)' AS when_checked,
(
SELECT COUNT(*) FROM cyntexa_dev.cyntexa_logistics.all_shipments_unified VERSION AS OF 11 WHERE shipment_id = 'SH99999'
) AS bad_row_count;

### Step 4: Cross-Reference with Metadata (Find the Source File)

In [0]:
%sql
-- ============================================================
-- STEP 4: USE METADATA TO FIND THE EXACT SOURCE FILE
-- ============================================================

SELECT
    shipment_id,
    origin,
    destination,
    weight_kg,
    ship_date,
    source_file,           -- <-- This tells us WHICH file!
    source_path            -- <-- This tells us WHERE the file is!
FROM cyntexa_dev.cyntexa_logistics.all_shipments_unified
WHERE shipment_id = 'SH99999';